In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [2]:
to_predict_mens = pd.read_csv("to_predict_mens.csv")

In [3]:
to_predict_mens[(to_predict_mens["GameRound"] == 1) & (to_predict_mens["Season"] == 2024)][["t1_TeamName", "t2_TeamName", "final_odds"]].head()

,t1_TeamName,t2_TeamName,final_odds
1318,Arizona,Long Beach St,-20.0
1319,Creighton,Akron,-12.5
1320,Dayton,Nevada,1.0
1321,Duquesne,BYU,7.5
1322,Gonzaga,McNeese St,-6.5


In [4]:
pd.set_option('display.max_columns', None)
1345
to_predict_mens[(to_predict_mens["Season"] == 2024)][["Team1", "t1_TeamName", "t1_adj_margin"]].drop_duplicates().sort_values(by="t1_adj_margin")

,Team1,t1_TeamName,t1_adj_margin
1317,1212,Grambling,-11.277963
2696,1224,Howard,-8.255369
1315,1447,Wagner,-8.062748
2719,1391,Stetson,-6.125959
2698,1286,Montana St,-4.311703
...,...,...,...
1324,1235,Iowa St,28.618729
2721,1388,St Mary's CA,28.632492
2730,1120,Auburn,28.847706
1338,1163,Connecticut,29.752956


In [5]:
to_predict_mens[(to_predict_mens["GameRound"] == 1) & (to_predict_mens["Season"] == 2024)
                & (to_predict_mens["Team2"] == 1345)]

,Unnamed: 0,type,ID,Pred,Season,Team1,Team2,Outcome,Gender,margin,t1_TeamName,t1_FirstD1Season,t1_LastD1Season,t2_TeamName,t2_FirstD1Season,t2_LastD1Season,final_odds,GameRound,t1_FGM,t1_FGA,t1_FGM3,t1_FGA3,t1_OR,t1_Ast,t1_TO,t1_Stl,t1_PF,t1_FTA,t1_FTM,t1_PointDiff,t2_FGM,t2_FGA,t2_FGM3,t2_FGA3,t2_OR,t2_Ast,t2_TO,t2_Stl,t2_PF,t2_FTA,t2_FTM,t2_PointDiff,t1_OrdinalRank,t2_OrdinalRank,t1_Seed,t2_Seed,seed_diff,t1_adj_oe,t1_adj_de,t1_adj_margin,t2_adj_oe,t2_adj_de,t2_adj_margin,t1_final_rank,t2_final_rank,t1_top8_TO_stdev,t1_top5_PRPG!_median,t1_top3_DR_median,t1_top5_STL_cv,t1_top3_Min%_median,t1_top8_TS_gini,t1_top3_USG_gini,t1_top8_BPM_weighted_mean,t2_top8_TO_stdev,t2_top5_PRPG!_median,t2_top3_DR_median,t2_top5_STL_cv,t2_top3_Min%_median,t2_top8_TS_gini,t2_top3_USG_gini,t2_top8_BPM_weighted_mean
2726,2728,Historical,2024_1212_1345,NaN,2024,1212,1345,0,M,-28,Grambling,1985,2025,Purdue,1985,2025,25.5,1,22.580645,52.354839,5.322581,15.612903,7.741935,9.16129,12.741935,7.129032,16.16129,20.741935,14.645161,-3.741935,28.515152,58.393939,8.333333,20.424242,11.030303,18.393939,10.969697,5.666667,14.363636,25.0,18.030303,13.242424,25.0,2.0,16,1,15,101.155352,112.433315,-11.277963,127.87684,100.690725,27.186115,66.780628,94.318459,5.996874,1.1,9.6,0.41779,66.6,0.050342,0.073737,-1.223557,3.197045,3.2,16.1,0.590937,77.4,0.10143,0.115556,7.184446


### Prepare data 

In [19]:
def prepare_data(to_predict_mens, season):

    to_predict_mens_first_round_train = to_predict_mens[(to_predict_mens["GameRound"] == 1)
                                                        & (to_predict_mens.Season < season)
                                                        & (to_predict_mens.Season >= 2008)
                                                        ].copy()
    
    to_predict_mens_train = to_predict_mens[(to_predict_mens.Season < season)
                                            & (to_predict_mens.Season >= 2008)].copy()

    to_predict_mens_first_round_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound == 1)
                                                    & (to_predict_mens.final_odds.notnull())
                                                    ].copy()

    to_predict_mens_other_rounds_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound > 1)
                                                    ].copy()
    
    return to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test
    


### Statistics Model Training

In [20]:
def train_statistics_model(to_predict_mens_train, statistics_features):

    best_params = {"C": .1}
    model = LogisticRegression(**best_params)
    pipeline = make_pipeline(StandardScaler(), model)
    statistics_model = pipeline.fit(to_predict_mens_train[statistics_features], to_predict_mens_train["Outcome"])

    return statistics_model


### Inference

In [21]:
def inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, statistics_model_full, statistics_model_first_round, 
              statistics_model_full_features, statistics_model_first_round_features):

    pred_proba = statistics_model_first_round.predict_proba(to_predict_mens_first_round_test[statistics_model_first_round_features].copy())[:,1]
    to_predict_mens_first_round_test["Pred"] = pred_proba
   
    pred_proba = statistics_model_full.predict_proba(to_predict_mens_other_rounds_test[statistics_model_full_features].copy())[:,1]
    to_predict_mens_other_rounds_test["Pred"] = pred_proba

    mens_sub = pd.concat([to_predict_mens_first_round_test[["ID", "Pred"]],
            to_predict_mens_other_rounds_test[["ID", "Pred"]]], axis=0)
    
    return mens_sub



### Full Pipeline

In [25]:
def run(to_predict_mens, season, statistics_model_full_features, statistics_model_first_round_features):
    
    # get data
    to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test= prepare_data(to_predict_mens, season)
    
    # train
    statistics_model_full = train_statistics_model(to_predict_mens_train, statistics_model_full_features)
    statistics_model_first_round = train_statistics_model(to_predict_mens_first_round_train, statistics_model_first_round_features)

    # inference
    sub = inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, 
                    statistics_model_full, statistics_model_first_round,
                    statistics_model_full_features, statistics_model_first_round_features) 

    return sub
    

In [26]:
to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test= prepare_data(to_predict_mens, 2024)

In [27]:
statistics_model_full = [   't1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
    't1_top8_TO_stdev', 't2_top8_TO_stdev',
    't1_top5_PRPG!_median', 't2_top5_PRPG!_median',
    't1_top3_DR_median', 't2_top3_DR_median',
    't1_top5_STL_cv', 't2_top5_STL_cv',
    't1_top3_Min%_median', 't2_top3_Min%_median',
    't1_top8_TS_gini', 't2_top8_TS_gini',
    't1_top3_USG_gini', 't2_top3_USG_gini',
    't1_OrdinalRank', 't2_OrdinalRank',
    't1_adj_margin', 't2_adj_margin', 
    ]

statistics_model_first_round = ['t1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
    't1_top8_TO_stdev', 't2_top8_TO_stdev',
    't1_top5_PRPG!_median', 't2_top5_PRPG!_median',
    't1_top3_DR_median', 't2_top3_DR_median',
    't1_top5_STL_cv', 't2_top5_STL_cv',
    't1_top3_Min%_median', 't2_top3_Min%_median',
    't1_top8_TS_gini', 't2_top8_TS_gini',
    't1_top3_USG_gini', 't2_top3_USG_gini',
    't1_OrdinalRank', 't2_OrdinalRank'
    ,]

mens_sub = run(to_predict_mens, 2024, statistics_model_full, statistics_model_first_round)

### Womens

In [29]:
to_predict_women = pd.read_csv("to_predict_women.csv")

to_predict_women_train = to_predict_women[to_predict_women.Season != 2024] 
to_predict_women_test = to_predict_women[to_predict_women.Season == 2024] 


In [32]:
statistics_features = ['seed_diff', 't1_adj_margin', 't2_adj_margin']
best_params = {"C": .1}
model = LogisticRegression(**best_params)
pipeline = make_pipeline(StandardScaler(), model)
statistics_model = pipeline.fit(to_predict_women_train[statistics_features], to_predict_women_train["Outcome"])
pred_proba = statistics_model.predict_proba(to_predict_women_test[statistics_features].copy())[:,1]
to_predict_women_test["Pred_Baseline"] = pred_proba


/var/folders/h5/f91pbgmj0rj6v0ls3y8zc5l40000gn/T/ipykernel_72087/889996954.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_predict_women_test["Pred_Baseline"] = pred_proba


In [30]:
SEASON = 2024

features = ['seed_diff', 
    't1_adj_margin', 't2_adj_margin',
    't1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean']

best_params = {"C": .1}

model = LogisticRegression(**best_params)
pipeline = make_pipeline(StandardScaler(), model)

to_predict_mens_train = to_predict_mens[(to_predict_mens.Season < SEASON)
                                            & (to_predict_mens.Season >= 2008)].copy()

statistics_model = pipeline.fit(to_predict_mens_train[features], to_predict_mens_train["Outcome"])
pred_proba = statistics_model.predict_proba(to_predict_women_test[features].copy())[:,1]

to_predict_women_test["Pred_New"] = pred_proba


/var/folders/h5/f91pbgmj0rj6v0ls3y8zc5l40000gn/T/ipykernel_72087/3365922374.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_predict_women_test["Pred_New"] = pred_proba


In [33]:
to_predict_women_test["Pred"] = (to_predict_women_test["Pred_Baseline"] + to_predict_women_test["Pred_New"]) / 2

/var/folders/h5/f91pbgmj0rj6v0ls3y8zc5l40000gn/T/ipykernel_72087/1750293826.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_predict_women_test["Pred"] = (to_predict_women_test["Pred_Baseline"] + to_predict_women_test["Pred_New"]) / 2


In [34]:

womens_sub = to_predict_women_test[["ID", "Pred"]]


In [35]:
final_sub = pd.concat([mens_sub, womens_sub], axis=0)

In [161]:
final_sub.to_csv("traditional_format_sub.csv")